# SBS 뉴스 직접 수집 — Selenium Colab 통합본

부산일보 reference 코드의 Selenium 패턴을 SBS 사이트(`news.sbs.co.kr`)와 Colab 자동화에 맞춰 변환한 통합 노트북. URL 수집과 본문 수집이 한 노트북에 들어있다.

- 입력: `press_ranges` — 언론사(press) + 일자 범위
- 출력: `data/링크_{press}_{YYMMDD}_{YYMMDD}.json`, `data/본문_{press}_{YYMMDD}_{YYMMDD}.csv`
- 보조 출력: 일자별 수집 로그 JSON, 본문 재실패 URL JSON, 본문 체크포인트 JSON
- 특징: 부산일보 패턴 그대로 (`newsflash.do` 단일 URL × pageIdx 순회, Selenium driver.get, 끝 버튼 href에서 max page 추출, `div.w_news_list` 컨테이너 안만 추출)
- BS4 트랙(`SBS_직접_url_수집_colab.ipynb` + `SBS_직접_본문_수집_bs4_colab.ipynb`)이 약 5~10배 빠르므로 본수집은 BS4 우선 권장. 이 노트북은 Selenium fallback / reference 보존용.
- press 이름은 BS4 트랙(`SBS_direct`)과 구분 위해 `SBS_direct_sel` 사용.

In [ ]:
# Colab 환경 세팅 — Selenium, Google Chrome, pandas 설치
!wget -q -O /tmp/google-chrome.deb https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!apt-get install -y -q /tmp/google-chrome.deb
!pip install -q selenium pandas

In [ ]:
# Google Drive 마운트 — 중간에 끊겨도 데이터 보존
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Drive 안의 프로젝트 폴더로 이동
import os
PROJECT_DIR = '/content/drive/MyDrive/Text-data-Analysis_26-Spring/news'
os.chdir(PROJECT_DIR)
print(f'현재 작업 폴더: {os.getcwd()}')

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
import json
import os
import random
import re
import shutil
import time
from datetime import datetime, timedelta
from pathlib import Path

import pandas as pd

# 로컬/Colab 비교를 위해 User-Agent 고정
USER_AGENT = 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/147.0.0.0 Safari/537.36'

# 담당 언론사와 수집 일자 범위 지정 — 날짜 형식: 'YYYY.MM.DD'
# 이름은 BS4 트랙(SBS_direct)과 충돌 안 나게 SBS_direct_sel 사용
# 같은 일자라도 BS4 결과와 별 파일로 저장되어 비교/병행 가능
press_ranges = [
    {'press': 'SBS_direct_sel', 'start_date': '2026.05.05', 'end_date': '2026.05.11'},
]


# 일자 단위 작업 목록 생성
# start_date부터 end_date까지 하루씩 쪼개 press × date 조합으로 펼침
def build_daily_jobs(press_ranges):
    jobs = []
    for item in press_ranges:
        press = item['press']
        start = datetime.strptime(item['start_date'], '%Y.%m.%d')
        end = datetime.strptime(item['end_date'], '%Y.%m.%d')
        # 시작 일자가 끝 일자보다 늦으면 범위 자체가 잘못된 것이므로 즉시 중단
        if start > end:
            raise ValueError(f'시작 일자가 끝 일자보다 늦습니다: {item}')
        # 시작 일자부터 끝 일자까지 하루씩 이동하면서 작업 생성
        current = start
        while current <= end:
            jobs.append({'press': press, 'date': current.strftime('%Y.%m.%d')})
            current += timedelta(days=1)
    return jobs


# 생성된 jobs는 아래 URL/본문 수집 셀에서 순서대로 실행
jobs = build_daily_jobs(press_ranges)
print(f'총 작업 수: {len(jobs)}')
for job in jobs:
    print(job)

# 셀 3을 건너뛰고 실행해도 기본 프로젝트 경로를 사용할 수 있게 보완
try:
    PROJECT_DIR
except NameError:
    PROJECT_DIR = '/content/drive/MyDrive/Text-data-Analysis_26-Spring/news'

# 저장할 폴더 지정
# 링크 파일, 본문 CSV, 체크포인트, 재실패 목록이 모두 이 폴더에 저장
SAVE_DIR = Path(PROJECT_DIR) / 'notebook' / 'crawling' / 'data'
SAVE_DIR.mkdir(parents=True, exist_ok=True)
print(f'저장 위치: {SAVE_DIR}')

# Selenium driver 설정 — Colab headless
# Selenium fallback — BS4가 처리 못 하는 JS 렌더링 페이지용
options = Options()
options.add_argument(f'user-agent={USER_AGENT}')  # 요청 환경을 일정하게 유지하기 위해 User-Agent 고정
options.add_experimental_option('excludeSwitches', ['enable-automation'])  # 자동화 제어 관련 switch 제외
options.add_experimental_option('useAutomationExtension', False)  # Selenium 자동화 확장 비활성화
options.add_argument('--disable-blink-features=AutomationControlled')  # AutomationControlled 플래그 비활성화
options.add_argument('--headless=new')  # Colab은 GUI가 없으므로 새 headless 모드 사용
options.add_argument('--no-sandbox')  # Colab 컨테이너 환경에서 Chrome 실행 안정화
options.add_argument('--disable-dev-shm-usage')  # /dev/shm 용량 부족으로 Chrome이 죽는 문제 완화
options.add_argument('--disable-gpu')  # headless 환경에서 GPU 관련 오류 방지
options.add_argument('--window-size=1920,1080')  # headless에서도 일정한 화면 크기로 렌더링

# Colab chromium-browser 패키지는 snap 래퍼라 Selenium에서 자주 실패
# 설치 셀에서 받은 Google Chrome 사용, ChromeDriver는 Selenium Manager에 맡김
chrome_binary = shutil.which('google-chrome') or shutil.which('google-chrome-stable') or '/usr/bin/google-chrome'
if not os.path.exists(chrome_binary):
    raise FileNotFoundError('Google Chrome을 찾지 못했습니다. 설치 셀을 먼저 실행하세요.')
options.binary_location = chrome_binary
print(f'Chrome binary: {chrome_binary}')

# Chrome 드라이버 설정
# Selenium Manager가 현재 Chrome 버전에 맞는 ChromeDriver를 자동으로 찾거나 내려받음
service = Service()
driver = webdriver.Chrome(service=service, options=options)
# navigator.webdriver 플래그 제거 — 자동화 탐지 회피를 위해 새 페이지마다 주입
driver.execute_cdp_cmd('Page.addScriptToEvaluateOnNewDocument', {
    'source': 'Object.defineProperty(navigator, "webdriver", {get: () => undefined})'
})
print('Selenium driver 준비 완료')

## 1단계: URL 수집 (`newsflash.do` × pageIdx 순회)

In [ ]:
# 봇 탐지 방지를 위해 페이지/job 사이에 짧은 랜덤한 시간을 기다림
PAGE_PAUSE_RANGE_SEC = (0.6, 1.5)
JOB_PAUSE_RANGE_SEC = (5, 12)
SKIP_COMPLETED = True


# 랜덤 대기 후 로그 출력
def polite_sleep(label, pause_range):
    pause_sec = random.uniform(*pause_range)
    print(f'{label} {pause_sec:.1f}초 대기')
    time.sleep(pause_sec)


# 일자별 newsflash 리스트 URL 조립 (pageIdx로 페이지 순회)
def build_list_url(date_ymd, page=1):
    return f'https://news.sbs.co.kr/news/newsflash.do?pageDate={date_ymd}&pageIdx={page}'


# 하루치 SBS newsflash 페이지를 모두 순회하며 기사 URL 수집
# 부산일보 reference 패턴 그대로 — 단일 URL × pageIdx, "끝 페이지 보기" href에서 max page 추출
def collect_links_for_day(press, date, save_dir=SAVE_DIR, driver=driver):
    # 파일명 키로 쓸 일자 접미사 (예: 2026.05.05 -> 260505)
    date_ymd = date.replace('.', '')
    date_yymmdd = date_ymd[2:]
    # 출력: 일자별 링크 JSON
    links_save_path = save_dir / f'링크_{press}_{date_yymmdd}_{date_yymmdd}.json'
    # 보조 출력: 페이지별 수집 로그 JSON
    stats_save_path = save_dir / f'수집로그_{press}_{date_yymmdd}_{date_yymmdd}.json'

    # 최종 파일이 이미 있으면 같은 일자는 건너뜀 (재실행 시 idempotent)
    if SKIP_COMPLETED and links_save_path.exists():
        print(f'\n=== {press} / {date} URL 이미 완료됨, 건너뜀 ===')
        print(f'기존 파일: {links_save_path}')
        return links_save_path

    print(f'\n=== {press} / {date} URL 수집 시작 ===')
    started_at = time.time()

    # 첫 페이지에서 "끝 페이지 보기" 링크의 href로 max page 추출
    # 페이지 수가 미리 보장되지 않으므로 끝 버튼 href에서 pageIdx 정규식으로 뽑아옴
    driver.get(build_list_url(date_ymd, 1))
    try:
        end_link = driver.find_element(By.CSS_SELECTOR, 'a[title="끝 페이지 보기"]')
        end_href = end_link.get_attribute('href') or ''
        max_page = int(re.search(r'pageIdx=(\d+)', end_href).group(1))
    except Exception:
        # 끝 버튼이 없거나 파싱 실패 — 단일 페이지로 간주
        max_page = 1
    print(f'전체 페이지 수: {max_page}')

    all_links = set()
    page_stats = []

    # 1페이지부터 max_page까지 순회
    for page in range(1, max_page + 1):
        if page > 1:
            # 첫 페이지는 위에서 이미 driver.get 했으므로 2페이지부터 다시 요청
            polite_sleep(f'page {page} 받기 전', PAGE_PAUSE_RANGE_SEC)
            driver.get(build_list_url(date_ymd, page))

        # 본문 컨테이너(div.w_news_list) 안 endPage.do 링크만 (사이드바 제외)
        # 사이드바에도 endPage.do 링크가 섞여 있어 컨테이너 한정 필요
        a_elements = driver.find_elements(By.CSS_SELECTOR, 'div.w_news_list a[href*="endPage.do"]')
        page_links = set()
        for a in a_elements:
            href = a.get_attribute('href') or ''
            # news_id=N{숫자} 패턴만 정규화 — 트래킹 쿼리스트링 제거
            m = re.search(r'news_id=N(\d+)', href)
            if m:
                page_links.add(f'https://news.sbs.co.kr/news/endPage.do?news_id=N{m.group(1)}')

        # set 차집합으로 신규 건수만 카운트
        before = len(all_links)
        all_links.update(page_links)
        added = len(all_links) - before
        page_stats.append({'page': page, 'found': len(page_links), 'added': added, 'total': len(all_links)})
        # 진행 상황 확인용 코드
        print(f'page {page}/{max_page} — {len(page_links)}건 / 신규 {added} / 누적 {len(all_links)}')

    elapsed = round(time.time() - started_at, 2)
    sbs_links = sorted(all_links)

    # 일자별 링크 최종 저장
    with open(links_save_path, 'w', encoding='utf-8') as f:
        json.dump(sbs_links, f, ensure_ascii=False, indent=2)
    # 페이지별 수집 통계 저장 — 특정 일자가 유독 적게/많이 잡혔는지 점검용
    with open(stats_save_path, 'w', encoding='utf-8') as f:
        json.dump({
            'press': press, 'date': date, 'total': len(sbs_links),
            'max_page': max_page, 'elapsed_sec': elapsed, 'pages': page_stats,
        }, f, ensure_ascii=False, indent=2)

    print(f'URL 수집 완료 — 총 {len(sbs_links)}개, {elapsed}초')
    print(f'저장: {links_save_path}')
    return links_save_path


# 모든 job의 URL 수집 실행
# 한 일자가 실패해도 실패 목록에 기록하고 다음 일자로 넘어감
url_results = []
url_failures = []
for index, job in enumerate(jobs, start=1):
    print(f'\n[{index}/{len(jobs)}] URL 수집: {job}')
    try:
        # job 딕셔너리의 press/date를 collect_links_for_day 인자로 전달
        url_results.append(collect_links_for_day(**job))
    except Exception as exc:
        # 한 일자에서 오류가 나도 전체 작업이 멈추지 않도록 실패 정보만 저장
        url_failures.append({'job': job, 'error': repr(exc)})
        print(f'URL 수집 실패: {exc!r}')
    finally:
        if index < len(jobs):
            # 다음 일자 job으로 넘어가기 전 대기
            polite_sleep('다음 작업 전', JOB_PAUSE_RANGE_SEC)

print(f'\nURL 수집 완료 — 성공 {len(url_results)}개, 실패 {len(url_failures)}개')

## 2단계: 본문 수집 (`driver.get` + meta/text_area 추출)

In [ ]:
# 봇 탐지 방지를 위해 기사 사이에 짧은 랜덤한 시간을 기다림
ARTICLE_PAUSE_RANGE_SEC = (0.8, 1.8)
# 체크포인트 저장 간격 — N건마다 한 번씩 중간저장 파일 갱신
CHECKPOINT_INTERVAL = 100


# 기사 한 건에서 title/body/pubdate/category 추출 (Selenium fallback)
def extract_article_selenium(link, driver=driver):
    # 실제 SBS 뉴스 웹페이지로 이동
    driver.get(link)
    # 페이지 로딩 대기
    time.sleep(0.3)

    # 제목 추출하기 — og:title 메타
    title = driver.find_element(By.CSS_SELECTOR, 'meta[property="og:title"]').get_attribute('content') or ''
    # 본문 추출하기 — schema.org articleBody 컨테이너, 연속 공백을 한 칸으로 정리
    body_el = driver.find_element(By.CSS_SELECTOR, 'div.text_area[itemprop="articleBody"]')
    body = re.sub(r'\s+', ' ', body_el.text).strip()
    # 날짜 추출하기 — article:published_time (ISO 8601 게시 시각)
    pubdate = driver.find_element(By.CSS_SELECTOR, 'meta[property="article:published_time"]').get_attribute('content') or ''
    # 카테고리 추출하기 — article:section 메타
    category = driver.find_element(By.CSS_SELECTOR, 'meta[property="article:section"]').get_attribute('content') or ''

    # 제목/본문/날짜 중 하나라도 없으면 실패 — 호출부에서 err_idx로 기록
    if not title or not body or not pubdate:
        raise ValueError('title/body/pubdate 중 일부 추출 실패')

    return {'link': link, 'pubdate': pubdate, 'category': category, 'title': title, 'body': body}


# 추출한 링크에 직접 방문하여 크롤링 진행
# 체크포인트 기반 재개 + 오류 자동 1회 재시도 + 재실패 URL JSON 저장
def collect_bodies_for_day(press, date, save_dir=SAVE_DIR, driver=driver):
    # 파일명 키로 쓸 일자 접미사
    date_ymd = date.replace('.', '')
    date_yymmdd = date_ymd[2:]
    links_path = save_dir / f'링크_{press}_{date_yymmdd}_{date_yymmdd}.json'
    csv_path = save_dir / f'본문_{press}_{date_yymmdd}_{date_yymmdd}.csv'
    # 체크포인트: 중간 결과(all_results) + 오류 인덱스 + 다음 인덱스를 보관
    checkpoint_path = save_dir / f'체크포인트_본문_{press}_{date_yymmdd}_{date_yymmdd}.json'

    # 최종 CSV가 이미 있으면 같은 일자는 건너뜀 (재실행 시 idempotent)
    if SKIP_COMPLETED and csv_path.exists():
        print(f'\n=== {press} / {date} 본문 이미 완료됨, 건너뜀 ===')
        return csv_path

    if not links_path.exists():
        raise FileNotFoundError(f'링크 파일 없음 — URL 수집 셀을 먼저 실행하세요: {links_path}')

    # 일자별 링크 파일 불러오기
    with links_path.open('r', encoding='utf-8') as f:
        links = json.load(f)
    print(f'\n=== {press} / {date} 본문 수집 시작 — 링크 {len(links)}개 ===')

    # 이전에 중단된 작업이 있으면 이어받기 — next_i 인덱스 다음부터 시작
    if checkpoint_path.exists():
        with checkpoint_path.open('r', encoding='utf-8') as f:
            cp = json.load(f)
        # JSON은 dict 키를 문자열로 저장하므로 int로 다시 변환
        all_results = {int(k): v for k, v in cp.get('all_results', {}).items()}
        err_idx = cp.get('err_idx', [])
        i = cp.get('next_i', 0)
        print(f'체크포인트 — {i}번째부터 (이미 수집 {len(all_results)}건)')
    else:
        all_results = dict()
        err_idx = []
        i = 0

    # 본문 수집 메인 루프 — i 인덱스를 함께 들고 다녀 체크포인트와 동기화
    for link in links[i:]:
        try:
            all_results[i] = extract_article_selenium(link)
            # 진행 상황 확인용 코드
            print(f'[{i+1}/{len(links)}] \t {(i+1)/len(links)*100:.1f}% \t err={len(err_idx)}')
            i += 1
            # 중간저장 — N건마다 체크포인트 갱신해 중간 중단에도 진행 보존
            if i % CHECKPOINT_INTERVAL == 0:
                with checkpoint_path.open('w', encoding='utf-8') as f:
                    json.dump({'all_results': all_results, 'err_idx': err_idx, 'next_i': i},
                              f, ensure_ascii=False, indent=2)
                print(f'체크포인트 저장 — {i}건')
            polite_sleep('다음 기사 전', ARTICLE_PAUSE_RANGE_SEC)
        except Exception as exc:
            # 오류 인덱스를 err_idx에 누적해두고 즉시 체크포인트도 갱신
            print(f'오류 i={i}: {exc!r}')
            err_idx.append(i)
            i += 1
            with checkpoint_path.open('w', encoding='utf-8') as f:
                json.dump({'all_results': all_results, 'err_idx': err_idx, 'next_i': i},
                          f, ensure_ascii=False, indent=2)

    # 1차 오류 자동 재시도 — 일시적 네트워크 문제였던 경우를 한 번 더 시도
    if err_idx:
        print(f'\n오류 {len(err_idx)}건 재시도...')
        re_err = []
        for retry_i in err_idx:
            try:
                all_results[retry_i] = extract_article_selenium(links[retry_i])
                print(f'재시도 성공 — i={retry_i}')
                polite_sleep('다음 재시도 전', ARTICLE_PAUSE_RANGE_SEC)
            except Exception as exc:
                print(f'재실패 — i={retry_i}: {exc!r}')
                re_err.append(retry_i)
        # 재시도 후에도 실패한 인덱스만 남김
        err_idx = re_err
        print(f'재시도 완료 — 재실패 {len(err_idx)}건')

    # 수집한 정보들을 dataframe으로 변환
    df = pd.DataFrame(all_results).T
    if df.empty:
        raise ValueError('수집된 본문 데이터가 없습니다.')
    # 중복 제거 후 pubdate 기준 오래된 순으로 정렬
    df = df.drop_duplicates().reset_index(drop=True)
    df['pubdate'] = pd.to_datetime(df['pubdate'], errors='coerce')
    df = df.sort_values(by='pubdate')
    # CSV 저장 (BOM 포함 UTF-8로 Excel 호환)
    df.to_csv(csv_path, index=False, encoding='utf-8-sig')
    print(f'CSV 저장 — {csv_path}')

    if err_idx:
        # 재시도 후에도 실패한 URL은 별 JSON으로 떨어뜨려 추후 수동 점검 가능
        failed_path = save_dir / f'본문_재실패_{press}_{date_yymmdd}_{date_yymmdd}.json'
        with failed_path.open('w', encoding='utf-8') as f:
            json.dump({'err_idx': err_idx, 'links': [links[x] for x in err_idx]},
                      f, ensure_ascii=False, indent=2)
        print(f'재실패 목록 저장 — {failed_path}')
    elif checkpoint_path.exists():
        # 재실패가 없을 때만 체크포인트 삭제 — 실패가 남아있으면 디버깅용으로 보존
        checkpoint_path.unlink()

    print(f'본문 수집 완료 — {len(df)}건 / 오류 {len(err_idx)}건')
    return csv_path


# 모든 job의 본문 수집 실행 — 한 일자가 실패해도 다음 일자로 넘어감
body_results = []
body_failures = []
for index, job in enumerate(jobs, start=1):
    print(f'\n[{index}/{len(jobs)}] 본문 수집: {job}')
    try:
        body_results.append(collect_bodies_for_day(**job))
    except Exception as exc:
        body_failures.append({'job': job, 'error': repr(exc)})
        print(f'본문 수집 실패: {exc!r}')
    finally:
        if index < len(jobs):
            polite_sleep('다음 작업 전', JOB_PAUSE_RANGE_SEC)

print(f'\n전체 완료 — URL {len(url_results)}, 본문 {len(body_results)} / 실패 URL {len(url_failures)}, 본문 {len(body_failures)}')

In [ ]:
# 브라우저 창 닫기 — Selenium driver 종료해 Chrome 프로세스 정리
driver.quit()
print('driver.quit 완료')